# 03b — Build the 2-class `dataset_face_lp`

Derives the **face + license-plate only** dataset (`nc=2`) from the full multi-class
`dataset/` produced by `03`. This is what the shipped YOLO26n model is trained on.

**What it keeps:**
- **all tiles containing a face or license-plate**, with labels filtered to classes 0/1;
- a controlled fraction (~12 %) of **car/person-only tiles as hard-negative backgrounds**
  — real scenes without faces/plates teach the model not to fire on cars/people (better
  than synthetic empty tiles);
- only the first **N photometric variants per crop** (`VAR_KEEP`), so the whole dataset
  fits in RAM for `cache='ram'` training (the offline 20× augmentation is overkill — the
  scene diversity comes from the source photos, not the photometric copies).

Images are symlinked to the already-decoded `aug/images` (via `realpath`, so the link
points straight at the real file, not through `dataset/`). Leakage-freeness is inherited
from `03` (the split is by source photo).

In [ ]:
import os, shutil, random, re, glob, sys
sys.path.insert(0, "/home/jovyan/shared/s0598584/scripts")
import config as C   # central config: classes, background ratio, variants kept

ROOT = str(C.ROOT)
SRC = f"{ROOT}/dataset"; DST = f"{ROOT}/dataset_face_lp"
# Keep exactly the configured class ids (as strings, matching YOLO label files).
KEEP = {str(i) for i in sorted(C.CLASSES)}
BG_RATIO = C.BG_RATIO            # fraction of hard-negative background tiles
VAR_KEEP = C.PHOTO_VARIANTS_KEEP # keep photometric variants v0..v(N-1)
print("keeping class ids:", sorted(KEEP), "| names:", C.class_names())
rng = random.Random(42)
var_re = re.compile(r"_v(\d+)$")

def variant_ok(stem):
    m = var_re.search(stem)
    return (m is None) or (int(m.group(1)) < VAR_KEEP)

if os.path.exists(DST):
    shutil.rmtree(DST)
stats = {}
for split in ["train", "val", "test"]:
    sl = f"{SRC}/labels/{split}"; si = f"{SRC}/images/{split}"
    dl = f"{DST}/labels/{split}"; di = f"{DST}/images/{split}"
    os.makedirs(dl); os.makedirs(di)
    pos, other, filt = [], [], {}
    for fn in os.listdir(sl):
        stem = os.path.splitext(fn)[0]
        if not variant_ok(stem):
            continue
        with open(os.path.join(sl, fn)) as fh:
            lines = [l for l in fh.read().splitlines() if l.strip()]
        kept = [l for l in lines if l.split()[0] in KEEP]
        if kept:
            pos.append(fn); filt[fn] = kept
        else:
            other.append(fn)   # no target class -> background candidate
    n_bg = min(len(other), round(len(pos) * BG_RATIO / (1 - BG_RATIO)))
    bg = rng.sample(other, n_bg)
    per_class = {k: 0 for k in KEEP}
    for fn in pos:
        stem = os.path.splitext(fn)[0]
        with open(os.path.join(dl, fn), "w") as fh:
            fh.write("\n".join(filt[fn]) + "\n")
        for l in filt[fn]:
            per_class[l.split()[0]] += 1
        os.symlink(os.path.realpath(os.path.join(si, stem + ".png")), os.path.join(di, stem + ".png"))
    for fn in bg:
        stem = os.path.splitext(fn)[0]
        open(os.path.join(dl, fn), "w").close()   # empty label = background
        os.symlink(os.path.realpath(os.path.join(si, stem + ".png")), os.path.join(di, stem + ".png"))
    tot = len(pos) + len(bg)
    stats[split] = dict(total=tot, pos=len(pos), bg=len(bg), per_class=dict(per_class))
    print(f"{split}: total={tot} pos={len(pos)} bg={len(bg)} per_class={per_class}")

names = "\n".join(f"  {i}: {C.CLASSES[i]}" for i in sorted(C.CLASSES))
with open(f"{DST}/dataset.yaml", "w") as fh:
    fh.write(f"path: {DST}\ntrain: images/train\nval: images/val\ntest: images/test\n"
             f"nc: {C.num_classes()}\nnames:\n{names}\n")
print("\ndataset.yaml ->", f"{DST}/dataset.yaml", "| nc =", C.num_classes())

## Sanity checks

In [ ]:
# no label should reference a class outside the configured set, and symlinks must resolve
bad = sum(1 for s in ["train","val","test"] for f in os.listdir(f"{DST}/labels/{s}")
          for l in open(f"{DST}/labels/{s}/{f}") if l.strip() and l.split()[0] not in KEEP)
img0 = glob.glob(f"{DST}/images/train/*.png")[0]
ram_gb = (stats['train']['total'] + stats['val']['total']) * 1280*1280*3 / 1e9
print("labels with class outside config (must be 0):", bad)
print("sample symlink resolves:", os.path.exists(img0))
print(f"train+val tiles cached in RAM: {stats['train']['total']+stats['val']['total']} -> ~{ram_gb:.1f} GB")

## ✅ Done

`dataset_face_lp/` (nc=2) is ready. Continue with `04_batch_finder.ipynb`.